# 00. Helper functions (read-only walkthrough)

The analysis notebooks import these functions from `src/football/`:
- `common.py` — geometry (distance, angle, cone), cross-validation, bootstrap
- `pressure.py` — pressure metrics (Andrienko, Pressing Intensity, FS arrival time) and pass-frame loading
- `tracking.py` — extracting frames before an event from the tracking files

Each function is shown below exactly as it is in the module, followed by a small check.
**This notebook is for reading.** The module files remain the source: if a function is changed, change it in
`src/football/*.py` so every notebook picks it up.

In [1]:
import numpy as np
import pandas as pd

## 1. `common.py` — constants and geometry

In [2]:
GOAL_X, POST_Y = 52.5, 3.66      # goal line x (m) and half goal width (m)
KEY = ["game_id", "pe_id"]

### `football.common.in_triangle`
Which points lie inside triangle a–b–c (used for defenders inside the shooting cone).

In [3]:
def in_triangle(pts, a, b, c):
    def s(p1, p2, p3):
        return (p1[:, 0] - p3[0]) * (p2[1] - p3[1]) - (p2[0] - p3[0]) * (p1[:, 1] - p3[1])
    d1, d2, d3 = s(pts, a, b), s(pts, b, c), s(pts, c, a)
    neg = (d1 < 0) | (d2 < 0) | (d3 < 0)
    pos = (d1 > 0) | (d2 > 0) | (d3 > 0)
    return ~(neg & pos)


In [4]:
in_triangle(np.array([[0.2, 0.2], [2.0, 2.0]]), np.array([0, 0]), np.array([1, 0]), np.array([0, 1]))  # [True, False]

array([ True, False])

### `football.common.geom`
Shot geometry at a point: distance and angle to goal, defenders in the cone, nearest defender, goalkeeper distance.

In [5]:
def geom(p, defs, gk):
    """Shot geometry at point p. defs: (n,2) outfield defenders, gk: (2,) or None."""
    post1, post2 = np.array([GOAL_X, -POST_Y]), np.array([GOAL_X, POST_Y])
    v1, v2 = post1 - p, post2 - p
    angle = abs(np.arctan2(v1[0] * v2[1] - v1[1] * v2[0], v1 @ v2))
    dist = np.hypot(GOAL_X - p[0], p[1])
    n_cone = int(in_triangle(defs, p, post1, post2).sum()) if len(defs) else 0
    near_def = np.hypot(*(defs - p).T).min() if len(defs) else 30.0
    gk_dist = np.hypot(*(gk - p)) if gk is not None else 30.0
    return dict(dist=dist, angle=angle, n_cone=n_cone, near_def=near_def, gk_dist=gk_dist)


In [6]:
geom(np.array([GOAL_X - 11, 0.0]), np.empty((0, 2)), None)  # penalty spot: dist 11, angle 0.642 rad

{'dist': np.float64(11.0),
 'angle': np.float64(0.6424100014274466),
 'n_cone': 0,
 'near_def': 30.0,
 'gk_dist': 30.0}

### `football.common.seg_dist`
Distance of each point to a segment (defenders to a passing lane).

In [7]:
def seg_dist(pts, a, b):
    """Distance of each point to segment a-b."""
    ab = b - a
    t = np.clip(((pts - a) @ ab) / max(ab @ ab, 1e-9), 0, 1)
    proj = a + t[:, None] * ab
    return np.hypot(*(pts - proj).T)


In [8]:
seg_dist(np.array([[0.0, 1.0], [3.0, 0.0]]), np.array([-1.0, 0]), np.array([1.0, 0]))  # [1, 2]

array([1., 2.])

In [9]:
XG_COLS = ["dist", "angle", "n_cone", "near_def", "gk_dist", "header"]

### `football.common.xg_X`
Design matrix of the xG model.

In [10]:
def xg_X(df):
    """Design matrix for the xG model (geometry + header + log distance)."""
    X = df.reindex(columns=XG_COLS).astype(float).fillna(0.0)
    X["log_dist"] = np.log1p(X.dist)
    return X


In [11]:
xg_X(pd.DataFrame({"dist": [11.0], "angle": [0.64], "n_cone": [0], "near_def": [3.0], "gk_dist": [10.0]}))

,dist,angle,n_cone,near_def,gk_dist,header,log_dist
0,11.0,0.64,0.0,3.0,10.0,0.0,2.484907


In [12]:
from sklearn.model_selection import GroupKFold

### `football.common.cv_predict`
Out-of-fold predictions with game-grouped 5-fold CV (a game is never in train and test at once), averaged over shuffles.

In [13]:
def cv_predict(make_model, X, y, groups, seeds=(0, 1, 2, 3, 4), proba=True):
    """Out-of-fold predictions, game-grouped 5-fold CV, averaged over shuffled group splits."""
    pred = np.zeros(len(y))
    for seed in seeds:
        rng = np.random.RandomState(seed)
        g_perm = pd.Series(rng.permutation(groups.unique()), index=groups.unique())
        for tr, te in GroupKFold(5).split(X, groups=groups.map(g_perm)):
            m = make_model(seed)
            m.fit(X.iloc[tr], y.iloc[tr])
            out = m.predict_proba(X.iloc[te])[:, 1] if proba else m.predict(X.iloc[te])
            pred[te] += out / len(seeds)
    return pred


### `football.common.boot_ci`
95% bootstrap confidence interval of a metric.

In [14]:
def boot_ci(y, p, fn, n=1000, seed=0):
    """95% percentile bootstrap CI of metric fn(y, p)."""
    y = np.asarray(y)
    p = np.asarray(p)
    rng = np.random.RandomState(seed)
    v = []
    for _ in range(n):
        i = rng.randint(0, len(y), len(y))
        if len(np.unique(y[i])) == 2:
            v.append(fn(y[i], p[i]))
    return np.percentile(v, [2.5, 97.5])


### `football.common.boot_diff`
Paired bootstrap of the difference between two models on the same cases.

In [15]:
def boot_diff(y, p1, p2, fn, n=1000, seed=0):
    """Paired bootstrap of fn(y,p1) - fn(y,p2): mean, 95% CI, share of resamples <= 0."""
    y, p1, p2 = map(np.asarray, (y, p1, p2))
    rng = np.random.RandomState(seed)
    d = []
    for _ in range(n):
        i = rng.randint(0, len(y), len(y))
        if len(np.unique(y[i])) == 2:
            d.append(fn(y[i], p1[i]) - fn(y[i], p2[i]))
    d = np.array(d)
    return d.mean(), np.percentile(d, [2.5, 97.5]), (d <= 0).mean()


In [16]:
from sklearn.metrics import roc_auc_score
rng = np.random.RandomState(0); y = rng.randint(0, 2, 500); p1 = y + rng.normal(0, 1, 500); p2 = rng.normal(0, 1, 500)
boot_diff(y, p1, p2, roc_auc_score, n=200)  # (mean gain, 95% CI, share of resamples <= 0)

(np.float64(0.33357758293905077),
 array([0.27127278, 0.39198463]),
 np.float64(0.0))

## 2. `pressure.py` — pressure metrics

### `football.pressure.andrienko`
Andrienko et al. (2017): oval pressure zone, 9 m in front (goal side), 3 m behind, decay exponent 1.75.

In [17]:
def andrienko(pc, pd_):
    """Andrienko et al. (2017) pressure per defender.

    Pressure zone length L grows from D_BACK (defender behind the carrier) to D_FRONT
    (defender between carrier and goal); pressure = (1 - d/L)^Q inside the zone, else 0.
    """
    D_FRONT, D_BACK, Q = 9.0, 3.0, 1.75
    to_goal = np.array([GOAL_X, 0.0]) - pc
    to_def = pd_ - pc
    d = np.hypot(*to_def.T)
    cos = (to_def @ to_goal) / np.maximum(d * np.hypot(*to_goal), 1e-9)
    z = (1 + cos) / 2
    L = D_BACK + (D_FRONT - D_BACK) * (z ** 3 + 0.3 * z) / 1.3
    return np.where(d < L, (1 - np.minimum(d / L, 1)) ** Q, 0.0)


In [18]:
andrienko(np.zeros(2), np.array([[1.0, 0.0], [-1.0, 0.0], [20.0, 0.0]]))  # goal-side > behind > far (0)

array([0.81373517, 0.49185863, 0.        ])

### `football.pressure.time_to_intercept`
Bekkers (2025) time to intercept (reaction 0.7 s, max speed 12 m/s, carrier extrapolated 1 s, turning penalty).

In [19]:
def time_to_intercept(pc, vc, pd_, vd, reaction=0.7, vmax=12.0):
    """Bekkers (2025) time to intercept (unravelsports implementation).

    Target extrapolated 1 s ahead (pc + vc); presser keeps its velocity for `reaction` s, then runs
    at vmax; plus a turning penalty |v_d|*theta/pi for moving away from the target.
    """
    target = pc + vc
    r = pd_ + vd * reaction
    to_t = target - pd_
    sp = np.hypot(*vd.T)
    cos = (vd * to_t).sum(1) / np.maximum(sp * np.hypot(*to_t.T), 1e-9)
    theta = np.where(sp > 1e-6, np.arccos(np.clip(cos, -1, 1)), 0.0)
    return reaction + np.hypot(*(target - r).T) / vmax + sp * theta / np.pi


### `football.pressure.intercept_prob`
Probability of intercepting within 1.5 s (logistic, sigma 0.45) — Pressing Intensity per defender.

In [20]:
def intercept_prob(tti, threshold=1.5, sigma=0.45):
    """Logistic probability that the defender reaches the carrier within `threshold` s."""
    return 1 / (1 + np.exp(-np.pi / (np.sqrt(3) * sigma) * (threshold - tti)))


In [21]:
intercept_prob(np.array([0.8, 1.5, 2.5]))  # high, 0.5, low

array([0.94382483, 0.5       , 0.01745251])

### `football.pressure.fs_arrival_time`
Narizuka et al. (2026): minimum arrival time under the Fujimura–Sugihara motion model (alpha 1, vmax 10 m/s).

In [22]:
def fs_arrival_time(pc, pd_, vd, alpha=1.0, vmax=10.0, t_max=10.0, iters=40):
    """Narizuka et al. (2026) minimum-arrival time under the Fujimura-Sugihara motion model.

    Reachable set at time t: circle centred c(t) = p + v(1-e^{-at})/a with radius
    r(t) = vmax (t - (1-e^{-at})/a). Arrival time = smallest t with |x - c(t)| <= r(t) (bisection).
    """
    lo, hi = np.zeros(len(pd_)), np.full(len(pd_), t_max)
    for _ in range(iters):
        t = (lo + hi) / 2
        g = (1 - np.exp(-alpha * t)) / alpha
        c = pd_ + vd * g[:, None]
        reached = np.hypot(*(pc - c).T) <= vmax * (t - g)
        hi = np.where(reached, t, hi)
        lo = np.where(reached, lo, t)
    return hi


In [23]:
fs_arrival_time(np.zeros(2), np.array([[10.0, 0.0], [10.0, 0.0]]), np.array([[-5.0, 0], [5.0, 0]]))  # running towards arrives sooner

array([1.37337455, 2.35810036])

### `football.pressure.closing_speed`
How fast the defender–carrier distance shrinks (m/s, positive = closing).

In [24]:
def closing_speed(pc, vc, pd_, vd):
    """Rate at which the defender-carrier distance shrinks (m/s, positive = closing)."""
    u = pc - pd_
    u = u / np.maximum(np.hypot(*u.T), 1e-9)[:, None]
    return ((vd - vc) * u).sum(1)


### `football.pressure.defender_table`
All per-defender features for one pass.

In [25]:
def defender_table(pc, vc, pd_, vd):
    """Per-defender features used by the metrics and the learned presser model (literature defaults)."""
    d = np.hypot(*(pd_ - pc).T)
    to_goal = np.array([GOAL_X, 0.0]) - pc
    cos = ((pd_ - pc) @ to_goal) / np.maximum(d * np.hypot(*to_goal), 1e-9)
    tti = time_to_intercept(pc, vc, pd_, vd)
    return dict(dist=d, cos_goal=cos, closing=closing_speed(pc, vc, pd_, vd),
                speed=np.hypot(*vd.T), tti=tti, p_int=intercept_prob(tti),
                andr=andrienko(pc, pd_), arrival=fs_arrival_time(pc, pd_, vd))


In [26]:
pd.DataFrame(defender_table(np.zeros(2), np.zeros(2), np.array([[2.0, 0.0], [-3.0, 1.0]]), np.array([[-2.0, 0.0], [0.0, 0.0]]))).round(3)

,dist,cos_goal,closing,speed,tti,p_int,andr,arrival
0,2.000,1.000,2.0,2.0,0.750,0.954,0.644,0.528
1,3.162,-0.949,0.0,0.0,0.964,0.897,0.000,0.916


### `football.pressure.pass_frames`
Raw positions at the pass frame + velocities from smoothed tracking (0.2 s difference).

In [27]:
def pass_frames(out_dir, keys, fps=29.97, lag=6):
    """Raw event-frame positions (players.csv.gz) + velocities from smoothed tracking (pass_frames_smoothed.csv.gz).

    keys: DataFrame with game_id, pe_id of the passes to keep. Returns one row per player per pass.
    """
    import os
    import pandas as pd
    k = ["game_id", "pe_id"]
    idx = keys.set_index(k).index
    pl = pd.read_csv(os.path.join(out_dir, "players.csv.gz"))
    raw = pl[pl.set_index(k).index.isin(idx)][k + ["player_id", "atk", "x", "y", "speed"]]
    fr = pd.read_csv(os.path.join(out_dir, "pass_frames_smoothed.csv.gz"))
    s0 = fr[fr.offset_frames == 0][k + ["player_id", "x", "y"]]
    s6 = fr[fr.offset_frames == lag][k + ["player_id", "x", "y"]]
    vel = s0.merge(s6, on=k + ["player_id"], suffixes=("", "6"))
    vel["vx"], vel["vy"] = (vel.x - vel.x6) * fps / lag, (vel.y - vel.y6) * fps / lag
    vel = vel.drop_duplicates(k + ["player_id"])[k + ["player_id", "vx", "vy"]]
    return raw.merge(vel, on=k + ["player_id"], how="left").fillna({"vx": 0.0, "vy": 0.0})


## 3. `tracking.py` — frames before an event

In [28]:
import bz2, json, os
from collections import deque
DATA_DIR = os.path.join("..", "data", "World_cup")

### `football.tracking.pre_event_frames`
Stream one game's tracking file and keep player positions N frames before each requested event.

In [29]:
def pre_event_frames(game_id, pe_ids, offsets_frames=(15, 30, 60), smoothed=False):
    """Player rows at `offset` frames before each possession event in `pe_ids`.

    Offset 0 = the event frame itself. `smoothed=True` reads the Kalman-smoothed
    coordinates (homePlayersSmoothed / awayPlayersSmoothed), better for velocities.
    Coordinates are flipped with the event's teamAttackingDirection so the team in
    possession attacks towards +x (same convention as 01_build_events).
    Returns a list of dicts: game_id, pe_id, offset_frames, player_id, atk, x, y, visible.
    """
    events = json.load(open(os.path.join(DATA_DIR, "Event Data", f"{game_id}_event.json"), encoding="utf-8"))
    meta = {}
    for e in events:
        if e["possessionEventId"] not in pe_ids:
            continue
        ge = e.get("gameEvents") or {}
        jersey = {}
        for side in ("homePlayers", "awayPlayers"):
            for p in e.get(side) or []:
                jersey[(side, str(p.get("jerseyNum")))] = p.get("playerId")
        meta[e["possessionEventId"]] = dict(
            flip=(e.get("stadiumMetadata") or {}).get("teamAttackingDirection") == "L",
            atk_home=ge.get("homeTeam") is True, jersey=jersey)

    max_off = max(offsets_frames)
    buf = deque(maxlen=max_off + 1)
    rows, done = [], set()
    path = os.path.join(DATA_DIR, "Tracking Data", f"{game_id}.jsonl_tracking.bz2")
    with bz2.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            fr = json.loads(line)
            buf.append(fr)
            pid = fr.get("possession_event_id")
            if pid is None or int(pid) not in meta or int(pid) in done:
                continue
            pid = int(pid)
            done.add(pid)
            m = meta[pid]
            fn = fr["frameNum"]
            by_num = {b["frameNum"]: b for b in buf}
            for off in offsets_frames:
                b = by_num.get(fn - off)
                if b is None or b.get("period") != fr.get("period"):
                    continue
                for side, atk in (("homePlayers", m["atk_home"]), ("awayPlayers", not m["atk_home"])):
                    for p in b.get(side + ("Smoothed" if smoothed else "")) or []:
                        if p.get("x") is None:
                            continue
                        x, y = (-p["x"], -p["y"]) if m["flip"] else (p["x"], p["y"])
                        rows.append(dict(game_id=game_id, pe_id=pid, offset_frames=off,
                                         player_id=m["jersey"].get((side, str(p.get("jerseyNum")))),
                                         atk=atk, x=x, y=y, visible=p.get("visibility") == "VISIBLE"))
    return rows
